# Monday, hands on: build Anand's suite yourself
#
# > "Nothing a person can mistype."
#
# Six queries, each with one comment line stating its question and its denominator. Replace every
# `__TODO__` and run the cell. The checks tell you whether a step worked without opening the
# solution.

In [ ]:
import pathlib
import sys

root = next(p for p in pathlib.Path.cwd().resolve().parents if (p / "scripts" / "c2kit.py").exists())
sys.path.insert(0, str(root / "scripts"))
import c2kit as kit

conn = kit.connect()
kit.flow(["connect", "filter", "group", "name the steps", "ship the suite"], lit=[0],
         title="Where you are")

## 1. The handshake
#
# Before anything else: how big is the book, and does it match what the platform lead said?

In [ ]:
n = kit.sql("__TODO1__", conn=conn)[0]
kit.check("the warehouse holds a thousand orders", list(n.values())[0] == 1000, str(n))

## 2. The largest orders
#
# Last week's first finding was the order that pulled the mean away from the median. One statement
# now. Give it an order, or the five rows you get back are any five rows.

In [ ]:
top = kit.sql("__TODO2__", conn=conn)
kit.check("five rows came back", len(top) == 5, f"{len(top)} rows")
kit.check("they are sorted from largest down",
          all(top[i]["amount"] >= top[i + 1]["amount"] for i in range(len(top) - 1)))

## 3. Per quarter
#
# Revenue and order count for each quarter, one row each.

In [ ]:
q = kit.sql("__TODO3__", conn=conn)
kit.check("one row per quarter", len(q) == 2, f"{len(q)} rows")
kit.table(list(q[0]) if q else ["result"], [list(r.values()) for r in q],
          caption="The two quarters")

## 4. Meet the error on purpose
#
# Ask for segment and channel while grouping by segment alone. Read the message, then fix it two
# ways and notice they answer two different questions.

In [ ]:
try:
    kit.sql("__TODO4__", conn=conn)
    print("no error: check that you grouped by segment alone")
except Exception as e:
    conn.rollback()
    print(str(e).strip().splitlines()[0])

## 5. Frequency, the branch that moved
#
# Orders per customer, per segment, per quarter. The denominator is customers who ordered in that
# quarter. Write that down in the comment before you write the SQL.

In [ ]:
freq = kit.sql("__TODO5__", conn=conn)
kit.check("eight rows, four segments across two quarters", len(freq) == 8, f"{len(freq)} rows")
kit.ladder(["orders", "per segment", "per quarter", "divided by customers"],
           lit=[3], title="Building the frequency number")

## 6. The comparison, as named steps
#
# Two CTEs, joined on segment, one row out per segment carrying the change.

In [ ]:
change = kit.sql("__TODO6__", conn=conn)
kit.check("one row per segment", len(change) == 4, f"{len(change)} rows")
worst = min(change, key=lambda r: float(list(r.values())[-1]))
kit.check("the steepest fall is Retail-Plus", worst["segment"] == "Retail-Plus", str(worst))
kit.tree({"label": "revenue moved", "branches": [
    ("customers?", {"label": "flat"}),
    ("frequency?", {"label": "fell", "branches": [
        ("which segment?", {"label": "Retail-Plus"})]}),
    ("basket?", {"label": "flat"})]},
    taken=["frequency?", "which segment?"],
    title="Which branch the suite points at")

## 7. Ship it
#
# Write the six queries into `sql/C2_W02_D01_02_monday_suite_STUDENT.sql` with their comment lines.
# The comment states the question and the denominator, and it is what Anand's analyst reads first.

In [ ]:
kit.decision_ladder(["a number in a message", "a spreadsheet you email",
                     "a query anybody can run", "a scheduled job"], cut_at=2,
                    title="What Anand asked for")
kit.check_summary()